## PCL Exchange: Full Implemented Feature Demo

This notebook demonstrates currently implemented features. Last updated 2026-09-09.

### End-to-End Flow

1. Create sender identity and signing keys.
2. Build a `request_measurement` message with full routing metadata.
3. Sign and inspect envelope fields, including content digest.
4. Validate through receiver pipeline (`parse_and_validate_crate`).
5. Exercise negative paths (signature tamper, schema invalid, SHACL invalid).
6. Build `ack` and `nack` protocol responses.
7. Demonstrate callback retries with a local mocked HTTP session.
8. Validate semantic shapes for all five domain actions.

In [ ]:
import copy
import json
import uuid
from datetime import datetime, timedelta, timezone

import requests
from jwcrypto import jwk

from pcl_exchange.builder import (
    PCLMessageBuilder,
    build_ack,
    build_nack,
)
from pcl_exchange.callback import CallbackClient
from pcl_exchange.crypto import Signer, Verifier
from pcl_exchange.models import PCLEnvelope, PCLErrorCode
from pcl_exchange.receiver import parse_and_validate_crate
from pcl_exchange.validation import validate_semantics, validate_structure


def extract_envelope(crate_dict):
    return next(item for item in crate_dict["@graph"] if item.get("@id") == "#envelope")


def extract_content(crate_dict):
    env = extract_envelope(crate_dict)
    content_ref = env["contentRef"]["@id"] if isinstance(env["contentRef"], dict) else env["contentRef"]
    return next(item for item in crate_dict["@graph"] if item.get("@id") == content_ref)


def resolver_from_single_key(public_key):
    def _resolver(sender_id):
        return public_key

    return _resolver


SENDER_ROR = "https://ror.org/0sndfake1"
RECEIVER_ROR = "https://ror.org/0rcvfake1"
ORIGIN_ORCID = "https://orcid.org/0000-0000-0000-0000"
PROJECT_DOI = "doi:10.5072/project.0001"
DATASET_DOI = "doi:10.5072/dataset.0001"
SAMPLE_IGSN = "igsn:JHABOX00000"

CONTENT_CONTEXT = [
    "https://w3id.org/ro/crate/1.1/context",
    {
        "prov": "http://www.w3.org/ns/prov#",
        "qudt": "http://qudt.org/schema/qudt/",
        "parameter": "http://schema.org/parameter",
        "unitText": "http://schema.org/unitText",
        "sha256": "http://schema.org/sha256",
        "generatedAtTime": {
            "@id": "http://www.w3.org/ns/prov#generatedAtTime",
            "@type": "http://www.w3.org/2001/XMLSchema#dateTime",
        },
    },
]


def build_signed_action_crate(sender_key, action, schema_uri, content, capabilities):
    envelope = {
        "@id": "#envelope",
        "@type": "PCLActionEnvelope",
        "profile": "https://w3id.org/pcl-profile/action/v1",
        "identifier": f"urn:uuid:{uuid.uuid4()}",
        "dateCreated": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "sender": {"@id": SENDER_ROR},
        "receiver": {"@id": RECEIVER_ROR},
        "schema": schema_uri,
        "action": action,
        "contentRef": {"@id": "#content"},
        "project": PROJECT_DOI,
        "sample": SAMPLE_IGSN,
        "capabilities": capabilities,
    }
    envelope["authz"] = {"type": "DetachedJWS", "jws": Signer(sender_key).sign(envelope)}
    return {"@context": CONTENT_CONTEXT, "@graph": [envelope, content]}


def summarize_result(name, result):
    print(f"{name}: valid={result.valid}, errors={[e.code for e in result.errors]}, shacl_report={'present' if result.shacl_report else 'none'}")

### 1) Establish Sender Identity

Generate an Ed25519 keypair for signing and verification in this demo.

The public key is what a receiver would obtain from sender identity infrastructure; this notebook uses an in-memory resolver for reproducibility.

In [ ]:
sender_key = jwk.JWK.generate(kty="OKP", crv="Ed25519")
sender_public_jwk = sender_key.export(private_key=False)

print(f"Sender key thumbprint: {sender_key.thumbprint()}")
print("Public JWK material is available for receiver verification.")

### 2) Build Request Message With Routing Metadata

Build a `request_measurement` payload and set routing/coordination fields that are part of the implemented envelope contract.

In [ ]:
builder = PCLMessageBuilder(
    sender_id=SENDER_ROR,
    receiver_id=RECEIVER_ROR,
)

builder.set_content(
    instrument="urn:aimd:instrument:proto-xrd-01",
    sample=SAMPLE_IGSN,
    method="urn:aimd:method:xrd:powder:theta-2theta:v1",
    params={
        "scan_range": {"val": "10 90", "unit": "deg 2theta"},
        "step": {"val": 0.02, "unit": "deg"},
    },
)
builder.add_capability("xrd.powder.theta-2theta")

deadline = datetime.now(timezone.utc) + timedelta(minutes=30)
builder.set_respond_to("https://example.org/hooks/status")
builder.set_correlation_id("pcl-req-00042")
builder.set_idempotency_key("pcl-req-00042-v1")
builder.set_ttl("PT30M")
builder.set_deadline(deadline)
builder.set_priority(5)
builder.set_protocol_version("1.1")
builder.set_schema(
    "https://w3id.org/pcl-schema/measure-request/v1.0",
    schema_hash={"alg": "sha256", "value": "b" * 64},
)

print("Builder configured for request_measurement with routing metadata.")

### 3) Sign, Serialize, and Inspect Envelope

Signing seals the builder to prevent post-sign mutation.

The envelope also carries `contentDigest`, which is computed from canonical payload JSON.

In [ ]:
signer = Signer(private_key=sender_key)
builder.sign(signer)
message_wire_format = builder.build().to_json()
message_data = json.loads(message_wire_format)
envelope_dict = extract_envelope(message_data)

print(f"Serialized message size: {len(message_wire_format)} bytes")
print("Envelope snapshot:")
for field in [
    "action",
    "schema",
    "respondTo",
    "correlationId",
    "idempotencyKey",
    "ttl",
    "deadline",
    "priority",
    "protocolVersion",
    "schemaHash",
]:
    print(f"- {field}: {envelope_dict.get(field)}")

print("\ncontentDigest:")
print(envelope_dict.get("contentDigest"))

try:
    builder.add_capability("late.mutation.should.fail")
except RuntimeError as exc:
    print(f"\nPost-sign mutation blocked as expected: {exc}")

### 4) Receiver Validation Pipeline (Happy Path)

Use `parse_and_validate_crate` to run structure, signature, and SHACL checks in the implemented receiver order.

In [ ]:
resolver = resolver_from_single_key(sender_key)
result_ok = parse_and_validate_crate(message_data, resolver)

print(f"valid: {result_ok.valid}")
print(f"errors: {[e.code for e in result_ok.errors]}")
print(f"shacl_report present: {result_ok.shacl_report is not None}")

verifier = Verifier(public_key=sender_key)
print(f"Direct verifier check: {verifier.verify(envelope_dict)}")

### 5) Negative Path
#### Signature Tamper

Mutate the envelope after signing. Signature should fail, and SHACL should be skipped in receiver pipeline.

In [ ]:
tampered_signature = copy.deepcopy(message_data)
extract_envelope(tampered_signature)["capabilities"] = ["tampered.capability"]

result_sig_bad = parse_and_validate_crate(tampered_signature, resolver)
summarize_result("tampered_signature", result_sig_bad)

#### Schema Invalid

Remove required envelope data to trigger schema failure. Receiver should report schema/signature issues and skip SHACL.

In [ ]:
schema_broken = copy.deepcopy(message_data)
del extract_envelope(schema_broken)["sender"]

result_schema_bad = parse_and_validate_crate(schema_broken, resolver)
summarize_result("schema_broken", result_schema_bad)

print("Detailed errors:")
for err in result_schema_bad.errors:
    print(f"- {err.code}: {err.message[:120]}")

#### SHACL Invalid While Signature/Schema Pass

For `register_data`, keep envelope valid and signed, but omit required `distribution` in content. This should produce `SHACL_VALIDATION_ERROR`.

In [ ]:
bad_register_content = {
    "@id": "#content",
    "@type": "Dataset",
    "name": "XRD Run 42 Results",
    "identifier": DATASET_DOI,
    "isPartOf": {"identifier": PROJECT_DOI},
    "about": {"identifier": SAMPLE_IGSN},
    "prov:wasAttributedTo": {"@id": SENDER_ROR},
    "generatedAtTime": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
}
bad_register_crate = build_signed_action_crate(
    sender_key,
    action="register_data",
    schema_uri="https://w3id.org/pcl-schema/register-data/v1.0",
    content=bad_register_content,
    capabilities=["data.register"],
)

result_shacl_bad = parse_and_validate_crate(bad_register_crate, resolver)
summarize_result("register_data_missing_distribution", result_shacl_bad)
if result_shacl_bad.errors:
    print(result_shacl_bad.errors[-1].code)

### 6) Protocol Responses

Use `build_ack` and `build_nack` to construct protocol-level responses.

- `ack` has no content shape and points `contentRef` to `#none`.
- `nack` points `contentRef` to `#error` and carries a structured `PCLError` payload.

In [ ]:
envelope_model = PCLEnvelope(**extract_envelope(message_data))

ack_envelope = build_ack(envelope_model, job_id="job-12345678")
nack_envelope, nack_error = build_nack(
    envelope_model,
    code=PCLErrorCode.CAPABILITY_MISMATCH,
    reason="Requested mode unavailable",
)

print("ACK fields:")
print(f"- action: {ack_envelope.action}")
print(f"- sender -> receiver swapped: {ack_envelope.sender} -> {ack_envelope.receiver}")
print(f"- correlationId: {ack_envelope.correlation_id}")
print(f"- contentRef: {ack_envelope.content_ref}")

print("\nNACK fields:")
print(f"- action: {nack_envelope.action}")
print(f"- contentRef: {nack_envelope.content_ref}")
print(f"- error code: {nack_error.code}")
print(f"- error reason: {nack_error.reason}")

### 7) Callback Delivery

Demonstrate callback behaviors:
- immediate success
- transient failures then success
- transient failures until retry exhaustion

In [ ]:
class FakeResponse:
    def __init__(self, status_code):
        self.status_code = status_code

    @property
    def ok(self):
        return 200 <= self.status_code < 300


class FakeSession:
    def __init__(self, status_codes):
        self.status_codes = list(status_codes)
        self.calls = 0

    def post(self, url, data, headers, timeout):
        self.calls += 1
        idx = min(self.calls - 1, len(self.status_codes) - 1)
        return FakeResponse(self.status_codes[idx])


callback_url = "https://example.org/hooks/status"

client_success = CallbackClient(session=FakeSession([200]), max_attempts=3, backoff_base=0.0)
result_cb_success = client_success.send(callback_url, ack_envelope)
print(f"success_case: success={result_cb_success.success}, attempts={result_cb_success.attempts}, status={result_cb_success.status_code}")

client_retry_then_ok = CallbackClient(session=FakeSession([503, 503, 200]), max_attempts=3, backoff_base=0.0)
result_cb_retry_ok = client_retry_then_ok.send(callback_url, ack_envelope)
print(f"retry_then_ok: success={result_cb_retry_ok.success}, attempts={result_cb_retry_ok.attempts}, status={result_cb_retry_ok.status_code}")

client_exhausted = CallbackClient(session=FakeSession([503, 503, 503]), max_attempts=3, backoff_base=0.0)
result_cb_exhausted = client_exhausted.send(callback_url, ack_envelope)
print(
    "exhausted: "
    f"success={result_cb_exhausted.success}, attempts={result_cb_exhausted.attempts}, "
    f"error_code={result_cb_exhausted.error.code if result_cb_exhausted.error else None}, "
    f"retriable={result_cb_exhausted.error.retriable if result_cb_exhausted.error else None}"
)

### 8) SHACL Coverage For Action Verbs

`request_measurement` is built with the builder API.

For the remaining actions, this demo hand-builds content nodes and signs envelopes, matching current implementation constraints.

JSON-LD caveats used below:
- `sha256` is explicitly mapped in local context.
- IRI-valued fields are emitted as `{"@id": "..."}`.
- For receiver SHACL path, `register_data.distribution` is nested inline.

In [ ]:
action_results = {}

# request_measurement via builder-generated message
request_struct_ok, request_struct_err = validate_structure(extract_envelope(message_data))
request_sem_ok, request_sem_err = validate_semantics(message_data, "shapes/request_measurement.ttl")
action_results["request_measurement"] = {
    "structure": request_struct_ok,
    "semantics": request_sem_ok,
    "note": request_struct_err or "ok",
}

# launch_workflow
launch_content = {
    "@id": "#content",
    "@type": "SoftwareSourceCode",
    "name": "Test Workflow",
    "programmingLanguage": "CWL v1.2",
    "codeRepository": {"@id": "https://example.org/workflows/main"},
    "parameter": [{"@type": "PropertyValue", "name": "scan_range", "value": "10 90"}],
}
launch_crate = build_signed_action_crate(
    sender_key,
    "launch_workflow",
    "https://w3id.org/pcl-schema/launch-workflow/v1.0",
    launch_content,
    ["workflow.cwl.launch"],
)
action_results["launch_workflow"] = {
    "receiver_valid": parse_and_validate_crate(launch_crate, resolver).valid,
}

# register_data (inline distribution for receiver SHACL sub-graph)
register_content = {
    "@id": "#content",
    "@type": "Dataset",
    "name": "XRD Run 42 Results",
    "identifier": DATASET_DOI,
    "isPartOf": {"identifier": PROJECT_DOI},
    "about": {"identifier": SAMPLE_IGSN},
    "prov:wasAttributedTo": {"@id": SENDER_ROR},
    "generatedAtTime": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
    "distribution": {
        "@type": "DataDownload",
        "contentUrl": {"@id": "https://example.org/data/run-42.zip"},
        "encodingFormat": "application/zip",
        "sha256": "a" * 64,
    },
}
register_crate = build_signed_action_crate(
    sender_key,
    "register_data",
    "https://w3id.org/pcl-schema/register-data/v1.0",
    register_content,
    ["data.register"],
)
action_results["register_data"] = {
    "receiver_valid": parse_and_validate_crate(register_crate, resolver).valid,
}

# update_metadata
update_content = {
    "@id": "#content",
    "@type": "UpdateAction",
    "object": {"@id": "https://example.org/datasets/42"},
    "prov:wasAttributedTo": {"@id": SENDER_ROR},
    "parameter": [{"@type": "PropertyValue", "name": "description", "value": "Updated description text"}],
}
update_crate = build_signed_action_crate(
    sender_key,
    "update_metadata",
    "https://w3id.org/pcl-schema/update-metadata/v1.0",
    update_content,
    ["data.update"],
)
action_results["update_metadata"] = {
    "receiver_valid": parse_and_validate_crate(update_crate, resolver).valid,
}

# cancel_job
cancel_content = {
    "@id": "#content",
    "@type": "Action",
    "object": {
        "@id": "#cancel-target",
        "@type": "PropertyValue",
        "name": "correlationId",
        "value": "pcl-req-00042",
    },
    "prov:wasAttributedTo": {"@id": SENDER_ROR},
}
cancel_crate = build_signed_action_crate(
    sender_key,
    "cancel_job",
    "https://w3id.org/pcl-schema/cancel-job/v1.0",
    cancel_content,
    ["job.cancel"],
)
action_results["cancel_job"] = {
    "receiver_valid": parse_and_validate_crate(cancel_crate, resolver).valid,
}

print("Action validation summary:")
for action_name, outcome in action_results.items():
    print(f"- {action_name}: {outcome}")